In [ ]:
from langchain.chat_models import ChatAnthropic
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains import ConversationChain
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

# 1. Base LLM
llm = ChatAnthropic(model="claude-3-haiku")

# 2. Short-term memory (auto-summarizes old turns)
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=2000  # static token budget
)

# 3. Long-term memory (RAG style with embeddings)
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts([], embeddings)

def add_to_vectorstore(text):
    vectorstore.add_texts([text])

def retrieve_relevant(query):
    return vectorstore.similarity_search(query, k=3)

# 4. Conversation chain with memory
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

# Example usage
user_input = "Help me debug Maven lifecycle errors"
# Add to vector DB for long-term recall
add_to_vectorstore(user_input)

# Retrieve relevant past facts
relevant_context = retrieve_relevant(user_input)

# Run conversation with rebased memory
response = conversation.run(input=user_input + "\nRelevant context:\n" + str(relevant_context))
print(response)
